<a href="https://colab.research.google.com/github/Jassmine11/cfpb-complaints-project/blob/main/Step_8_Scale_Your_ML_Prototype.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Explanation: mount Drive, set seeds, and define folder paths used by the pipeline.**bold text**

In [ ]:
# Cell 0 — setup
from google.colab import drive
drive.mount('/content/drive')

import os, time, json, hashlib, shutil
from pathlib import Path
import numpy as np, pandas as pd
import joblib

# Reproducibility
SEED = 42
np.random.seed(SEED)

# Paths (edit DRIVE_BASE if you prefer)
DRIVE_BASE = Path("/content/drive/MyDrive/capstone_artifacts")
CLEANED_DRIVE = DRIVE_BASE / "data" / "cleaned" / "cfpb_cleaned_step5.csv"
CLEANED_LOCAL = Path("data/cleaned/cfpb_cleaned_v1.csv")
EMB_DIR = Path("artifacts/embeddings")
EMB_DIR_DRIVE = DRIVE_BASE / "artifacts" / "embeddings"
FAISS_DIR = Path("artifacts/faiss")
MODELS_DIR = Path("models")
RESULTS_DIR = Path("step8_results")
LOGS_DIR = DRIVE_BASE / "logs"

for p in [EMB_DIR, FAISS_DIR, MODELS_DIR, RESULTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)

print("Paths set. Cleaned source (Drive):", CLEANED_DRIVE)


Mounted at /content/drive
Paths set. Cleaned source (Drive): /content/drive/MyDrive/capstone_artifacts/data/cleaned/cfpb_cleaned_step5.csv


Explanation: confirm cleaned CSV exists in Drive and copy to notebook working dir for streaming.**bold text**

In [ ]:
# Cell 1 — ensure cleaned data available locally
if CLEANED_DRIVE.exists():
    CLEANED_LOCAL.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(CLEANED_DRIVE, CLEANED_LOCAL)
    print("Copied cleaned CSV to local:", CLEANED_LOCAL)
else:
    raise FileNotFoundError(f"Cleaned file not found at {CLEANED_DRIVE}")
# Optional: show basic info
df_head = pd.read_csv(CLEANED_LOCAL, nrows=3)
print("Sample rows:\n", df_head)


Copied cleaned CSV to local: data/cleaned/cfpb_cleaned_v1.csv
Sample rows:
   Date received                                            Product  \
0    2019-08-29  Credit reporting, credit repair services, or o...   
1    2025-07-22                                        Credit card   
2    2025-06-09  Credit reporting or other personal consumer re...   

         Sub-product                                              Issue  \
0   Credit reporting  Problem with a credit reporting company's inve...   
1  Store credit card    Problem with a purchase shown on your statement   
2   Credit reporting  Problem with a company's investigation into an...   

                                           Sub-issue  \
0  Their investigation did not fix an error on yo...   
1  Card was charged for something you did not pur...   
2  Their investigation did not fix an error on yo...   

                        Consumer complaint narrative               Company  \
0  On XX/XX/XXXX, I received a personal

Explanation: read raw/cleaned CSV in chunks, apply light cleaning & PII redaction, write versioned cleaned CSV (out-of-core).**bold text**

In [ ]:
# Cell 2 — chunked ETL + redaction -> produces data/cleaned/cfpb_cleaned_v1.csv (idempotent append)
import re
SRC = str(CLEANED_LOCAL)   # using cleaned file as starting point; if starting raw use raw filename
OUT = CLEANED_LOCAL  # we overwrite local cleaned file in this flow (already copied)
CHUNK = 100000

def simple_redact(text):
    if not isinstance(text, str): return text
    text = re.sub(r'\b[\w\.-]+@[\w\.-]+\.\w{2,}\b','[EMAIL]', text)
    text = re.sub(r'\b(?:\+?\d[\d\-\s().]{7,}\d)\b','[PHONE]', text)
    text = re.sub(r'\b(?:\d{4,}\-?\d{4,}\-?\d{4,}\d?)\b','[ACCOUNT]', text)
    return text

# Read in chunks, apply minimal cleaning and redaction, append to new file (idempotent behavior: write fresh)
first = True
reader = pd.read_csv(SRC, chunksize=CHUNK)
for i,chunk in enumerate(reader):
    # minimal cleaning consistent with Step 5
    drop_cols = ["Consumer disputed?","Tags","Company public response","Consumer consent provided?"]
    chunk.drop(columns=[c for c in drop_cols if c in chunk.columns], inplace=True, errors='ignore')
    # keep only rows with narrative
    if "Consumer complaint narrative" not in chunk.columns:
        raise KeyError("Missing required column: Consumer complaint narrative")
    chunk = chunk[chunk["Consumer complaint narrative"].notna()].copy()
    chunk["Consumer complaint narrative"] = chunk["Consumer complaint narrative"].astype(str).str.strip().apply(simple_redact)
    for col in ["Sub-issue","Sub-product","State","ZIP code"]:
        if col in chunk.columns:
            chunk[col] = chunk[col].fillna("Unknown")
    # write
    mode = 'w' if first else 'a'
    header = first
    chunk.to_csv(OUT, mode=mode, header=header, index=False)
    first = False
    print(f"Processed chunk {i}, rows written: {len(chunk)}")
print("ETL/redaction complete — cleaned file:", OUT)
# copy provenance (md5) to Drive logs
def md5(fname):
    h=hashlib.md5()
    with open(fname,"rb") as f:
        for b in iter(lambda: f.read(8192), b""):
            h.update(b)
    return h.hexdigest()
meta = {"file": str(OUT), "md5": md5(OUT), "rows": sum(1 for _ in open(OUT)) - 1, "timestamp": time.time()}
json.dump(meta, open(RESULTS_DIR/"etl_provenance.json","w"), indent=2)
shutil.copy2(RESULTS_DIR/"etl_provenance.json", LOGS_DIR/"etl_provenance.json")
print("Saved provenance to Drive logs.")


Processed chunk 0, rows written: 26861
ETL/redaction complete — cleaned file: data/cleaned/cfpb_cleaned_v1.csv
Saved provenance to Drive logs.


Explanation: train a scalable TF-style classifier in streaming fashion without storing vocabulary.**bold text**

In [ ]:
# Complete streaming training cell (copy-paste and run)
from pathlib import Path
import pandas as pd, joblib, numpy as np
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.linear_model import SGDClassifier

TEXT_COL = "Consumer complaint narrative"
LABEL_COL = "Product"
CHUNK_TRAIN = 20000
SEED = 42

# 1) load full labels to compute manual balanced class weights
labels_full = pd.read_csv("data/cleaned/cfpb_cleaned_v1.csv", usecols=[LABEL_COL])[LABEL_COL].astype(str)
classes = labels_full.unique().tolist()
vc = labels_full.value_counts().to_dict()
total = len(labels_full)
num_classes = len(classes)
class_weights = {c: total / (num_classes * vc.get(c, 1)) for c in classes}
print("Computed class weights for", num_classes, "classes. Example:", list(class_weights.items())[:5])

# 2) init vectorizer and classifier
vec = HashingVectorizer(n_features=2**20, ngram_range=(1,2), alternate_sign=False)
clf = SGDClassifier(loss="log_loss", max_iter=1, tol=None, random_state=SEED, warm_start=True)

# 3) incremental training with per-sample weights
first = True
for i, chunk in enumerate(pd.read_csv("data/cleaned/cfpb_cleaned_v1.csv", usecols=[TEXT_COL, LABEL_COL], chunksize=CHUNK_TRAIN)):
    texts = chunk[TEXT_COL].astype(str).tolist()
    y = chunk[LABEL_COL].astype(str).values
    X = vec.transform(texts)
    sample_weight = np.array([class_weights[label] for label in y], dtype=float)
    if first:
        clf.partial_fit(X, y, classes=classes, sample_weight=sample_weight)
        first = False
    else:
        clf.partial_fit(X, y, sample_weight=sample_weight)
    print(f"Trained on chunk {i}, rows {len(chunk)}")

# 4) persist model
Path("models").mkdir(parents=True, exist_ok=True)
joblib.dump(clf, "models/sgd_hash_clf_v1.joblib")
print("Saved model: models/sgd_hash_clf_v1.joblib")



Computed class weights for 19 classes. Example: [('Credit reporting, credit repair services, or other personal consumer reports', 0.22627030123323674), ('Credit card', 1.606519138755981), ('Credit reporting or other personal consumer reports', 0.13004662331940606), ('Debt collection', 0.5094547178757705), ('Credit card or prepaid card', 1.531675885271141)]
Trained on chunk 0, rows 20000
Trained on chunk 1, rows 6861
Saved model: models/sgd_hash_clf_v1.joblib


Explanation: create dense embeddings in manageable chunks, save each chunk and its metadata for resume/indexing.

In [ ]:
# Cell 4 — batched embeddings (Sentence‑Transformers) with chunked saves
!pip install -q sentence-transformers faiss-cpu
from sentence_transformers import SentenceTransformer
BATCH = 256
EMB_MODEL = "all-MiniLM-L6-v2"
emb_model = SentenceTransformer(EMB_MODEL)

# read in chunks, encode, save per chunk
reader = pd.read_csv(OUT, usecols=["Complaint ID","Consumer complaint narrative"], chunksize=50000)
for i,chunk in enumerate(reader):
    texts = chunk["Consumer complaint narrative"].astype(str).tolist()
    emb = emb_model.encode(texts, batch_size=BATCH, show_progress_bar=True)
    np.save(EMB_DIR/f"emb_chunk_{i}.npy", emb.astype("float32"))
    chunk[["Complaint ID"]].to_csv(EMB_DIR/f"meta_chunk_{i}.csv", index=False)
    # copy to Drive for persistence
    EMB_DIR_DRIVE.mkdir(parents=True, exist_ok=True)
    shutil.copy2(EMB_DIR/f"emb_chunk_{i}.npy", EMB_DIR_DRIVE/f"emb_chunk_{i}.npy")
    shutil.copy2(EMB_DIR/f"meta_chunk_{i}.csv", EMB_DIR_DRIVE/f"meta_chunk_{i}.csv")
    print(f"Saved embedding chunk {i}, rows {len(texts)}")
print("Embeddings generation complete. Files saved locally and to Drive.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 71.9 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/105 [00:00<?, ?it/s]

Saved embedding chunk 0, rows 26861
Embeddings generation complete. Files saved locally and to Drive.


Explanation: assemble per-chunk embeddings into a single FAISS IVF index and save a mapping from index positions to complaint IDs.

In [ ]:
# Cell 5 — build FAISS index from saved embedding chunks
import faiss, glob, pandas as pd
emb_files = sorted(glob.glob(str(EMB_DIR/"emb_chunk_*.npy")))
d = np.load(emb_files[0]).shape[1]  # dimension
nlist = min(4096, max(64, len(emb_files)*10))  # heuristic
quantizer = faiss.IndexFlatIP(d)
index = faiss.IndexIVFFlat(quantizer, d, nlist, faiss.METRIC_INNER_PRODUCT)

# train on sample (first chunk)
train_emb = np.load(emb_files[0])
faiss.normalize_L2(train_emb)
index.train(train_emb)

global_offset = 0
meta_rows = []
for f in emb_files:
    arr = np.load(f)
    faiss.normalize_L2(arr)
    index.add(arr)
    # read corresponding meta
    meta_file = f.replace("emb_chunk_", "meta_chunk_").replace(".npy", ".csv")
    ids = pd.read_csv(meta_file)["Complaint ID"].tolist()
    for i, cid in enumerate(ids):
        meta_rows.append((global_offset + i, int(cid)))
    global_offset += arr.shape[0]
# save index & metadata
index_file = FAISS_DIR/"cfpb_ivf.index"
faiss.write_index(index, str(index_file))
pd.DataFrame(meta_rows, columns=["global_index","complaint_id"]).to_csv(FAISS_DIR/"faiss_meta.csv", index=False)
# copy to Drive for persistence
shutil.copy2(str(index_file), str(DRIVE_BASE / "artifacts" / "faiss_index.ivf"))
shutil.copy2(str(FAISS_DIR/"faiss_meta.csv"), str(DRIVE_BASE / "artifacts" / "faiss_meta.csv"))
print("FAISS index and metadata saved.")


FAISS index and metadata saved.


Explanation: example retrieval flow: embed query, normalize, retrieve top-k, map to complaint IDs and return redacted text from cleaned file.

In [ ]:
# Cell 6 — retrieval example
index = faiss.read_index(str(FAISS_DIR/"cfpb_ivf.index"))
meta_df = pd.read_csv(FAISS_DIR/"faiss_meta.csv")
# single query
query = "I was charged twice for a transaction; want refund"
q_emb = emb_model.encode([query])
faiss.normalize_L2(q_emb)
k = 5
D,I = index.search(q_emb.astype("float32"), k)
indices = I[0].tolist()
results = meta_df[meta_df["global_index"].isin(indices)].merge(pd.read_csv(OUT)[["Complaint ID","Consumer complaint narrative"]].astype({"Complaint ID":int}), left_on="complaint_id", right_on="Complaint ID", how="left")
print("Top retrieved complaint IDs and snippets:")
print(results[["complaint_id","Consumer complaint narrative"]].head())


Top retrieved complaint IDs and snippets:
   complaint_id                       Consumer complaint narrative
0       6391450  On XX/XX/2022 I was rebilled for one of the tw...
1      13088465  I was charged twice for the same hotel reserva...
2      12747141  On XX/XX/year>, I was charged {$220.00} for co...
3      11636887  I would like to be reimbursed, I would like a ...
4       7353225  In XX/XX/XXXX  I filed a chargeback for produc...


Explanation: ensure all artifacts, sizes, and checksums are persisted and logged for reproducibility.

In [ ]:
# Cell 7 — save key artifacts and experiment metadata
# Copy local artifacts (if not already copied)
shutil.copy2(str(CLEANED_LOCAL), str(DRIVE_BASE/"data"/"cleaned"/CLEANED_LOCAL.name))
shutil.copytree(str(EMB_DIR), str(DRIVE_BASE/"artifacts"/"embeddings"), dirs_exist_ok=True)
shutil.copy2(str(FAISS_DIR/"cfpb_ivf.index"), str(DRIVE_BASE/"artifacts"/"faiss_index.ivf"))
shutil.copy2(str(FAISS_DIR/"faiss_meta.csv"), str(DRIVE_BASE/"artifacts"/"faiss_meta.csv"))
shutil.copytree(str(MODELS_DIR), str(DRIVE_BASE/"models"), dirs_exist_ok=True)
shutil.copytree(str(RESULTS_DIR), str(DRIVE_BASE/"results"), dirs_exist_ok=True)

# Experiment metadata file
exp_meta = {
    "seed": SEED,
    "cleaned_data": str(DRIVE_BASE/"data"/"cleaned"/CLEANED_LOCAL.name),
    "cleaned_md5": md5(str(DRIVE_BASE/"data"/"cleaned"/CLEANED_LOCAL.name)),
    "n_rows": sum(1 for _ in open(CLEANED_LOCAL)) - 1,
    "emb_chunks": len(list(EMB_DIR.glob("emb_chunk_*.npy"))),
    "faiss_index": str(DRIVE_BASE/"artifacts"/"faiss_index.ivf"),
    "timestamp": time.time()
}
json.dump(exp_meta, open(RESULTS_DIR/"experiment_meta_step8.json","w"), indent=2)
shutil.copy2(RESULTS_DIR/"experiment_meta_step8.json", LOGS_DIR/"experiment_meta_step8.json")
print("Saved experiment metadata to Drive logs.")


Saved experiment metadata to Drive logs.


In [ ]:
#verify embidding and split
from pathlib import Path
import numpy as np, joblib
print("emb chunks:", sorted([p.name for p in Path("artifacts/embeddings").glob("emb_chunk_*.npy")]))
print("emb_train.npy exists:", Path("step7_results/emb_train.npy").exists())
print("emb_test.npy exists:", Path("step7_results/emb_test.npy").exists())
X_train, X_test, y_train, y_test, le = joblib.load("/content/drive/MyDrive/capstone_artifacts/artifacts/splits/split_and_le_step7.joblib")
print("len X_train, X_test:", len(X_train), len(X_test))


emb chunks: ['emb_chunk_0.npy']
emb_train.npy exists: False
emb_test.npy exists: False
len X_train, X_test: 21488 5373


In [ ]:
# Build emb_train.npy and emb_test.npy from emb_chunk_0 + meta_chunk_0 and saved splits
import glob, pandas as pd, numpy as np, joblib
from pathlib import Path

# Paths
EMB_DIR = Path("artifacts/embeddings")
CLEANED = Path("data/cleaned/cfpb_cleaned_v1.csv")   # local cleaned file
SPLIT_JOBLIB = "/content/drive/MyDrive/capstone_artifacts/artifacts/splits/split_and_le_step7.joblib"

# Load splits
X_train, X_test, y_train, y_test, le = joblib.load(SPLIT_JOBLIB)
print("len X_train, X_test:", len(X_train), len(X_test))

# Load embeddings and meta
emb_files = sorted(EMB_DIR.glob("emb_chunk_*.npy"))
meta_files = sorted(EMB_DIR.glob("meta_chunk_*.csv"))
if not emb_files or not meta_files:
    raise FileNotFoundError("No emb_chunk_*.npy or meta_chunk_*.csv found in artifacts/embeddings")

# Build id -> embedding map
id_to_emb = {}
for ef, mf in zip(emb_files, meta_files):
    em = np.load(ef)
    ids = pd.read_csv(mf)["Complaint ID"].astype(int).tolist()
    if len(ids) != em.shape[0]:
        print(f"Warning: length mismatch {mf} vs {ef}")
    for i, cid in enumerate(ids):
        id_to_emb[int(cid)] = em[i]

print("Total mapped embeddings:", len(id_to_emb))

# Build text -> Complaint ID mapping from cleaned CSV
df = pd.read_csv(CLEANED, usecols=["Complaint ID","Consumer complaint narrative"])
# Use stripped texts for more reliable matching
df["Consumer complaint narrative"] = df["Consumer complaint narrative"].astype(str).str.strip()
text_to_id = dict(zip(df["Consumer complaint narrative"].tolist(), df["Complaint ID"].astype(int).tolist()))
print("Built text->ID map with entries:", len(text_to_id))

# Map X_train/X_test texts to IDs
train_ids = [text_to_id.get(t.strip()) for t in X_train]
test_ids  = [text_to_id.get(t.strip()) for t in X_test]

miss_train = sum(1 for x in train_ids if x is None)
miss_test  = sum(1 for x in test_ids if x is None)
print("Missing mappings — train:", miss_train, "test:", miss_test)

# If many missing, print a few examples
if miss_train > 0:
    print("Sample missing train texts:", [X_train[i] for i,x in enumerate(train_ids) if x is None][:3])
if miss_test > 0:
    print("Sample missing test texts:", [X_test[i] for i,x in enumerate(test_ids) if x is None][:3])

# Build emb arrays skipping missing IDs
train_emb_list = [id_to_emb[cid] for cid in train_ids if cid is not None and cid in id_to_emb]
test_emb_list  = [id_to_emb[cid] for cid in test_ids  if cid is not None and cid in id_to_emb]

emb_train = np.vstack(train_emb_list) if train_emb_list else np.empty((0, np.load(emb_files[0]).shape[1]))
emb_test  = np.vstack(test_emb_list)  if test_emb_list  else np.empty((0, np.load(emb_files[0]).shape[1]))

print("Built emb_train shape:", emb_train.shape, "emb_test shape:", emb_test.shape)

# Save for reuse
Path("step7_results").mkdir(parents=True, exist_ok=True)
np.save("step7_results/emb_train.npy", emb_train)
np.save("step7_results/emb_test.npy", emb_test)
print("Saved emb_train.npy and emb_test.npy to step7_results/")


len X_train, X_test: 21488 5373
Total mapped embeddings: 26861
Built text->ID map with entries: 23361
Missing mappings — train: 556 test: 141
Sample missing train texts: ['It has come to my attention that the Company PORTFOLIO RECOV ASSOC has acquired the account due to material misrepresentation, fraud, and Identity Theft pursuant to 15 USC 1681a ( q ) ( 3 ) and RCW 9.35.020. The Original Creditor failed to legally obtain consent to sell, transfer, or disclose any account information with your company or any other non-affiliated third party as mandated by 15 USC 6802 of the GLBA. Failure to provide FULL disclosure of material information constitutes an unfair method of competition and unfair & deceptive acts of FRAUD. When fraud leads to the formation of a contract, the contract becomes unenforceable, and the defrauded party may void it upon discovery of the fraud. Thus, it is hereby notified that upon discovering FRAUD associated with the contract between the consumer and the origina

**Recreate aligned ID lists and save (repeat mapping, produce filtered id lists)**

In [ ]:
# Recreate mapping and save filtered train/test id lists
import glob, pandas as pd, numpy as np, joblib
from pathlib import Path

EMB_DIR = Path("artifacts/embeddings")
CLEANED = Path("data/cleaned/cfpb_cleaned_v1.csv")
SPLIT_JOBLIB = "/content/drive/MyDrive/capstone_artifacts/artifacts/splits/split_and_le_step7.joblib"

X_train, X_test, y_train, y_test, le = joblib.load(SPLIT_JOBLIB)

# load id->emb mapping
emb_files = sorted(EMB_DIR.glob("emb_chunk_*.npy"))
meta_files = sorted(EMB_DIR.glob("meta_chunk_*.csv"))
id_to_emb = {}
for ef, mf in zip(emb_files, meta_files):
    em = np.load(ef)
    ids = pd.read_csv(mf)["Complaint ID"].astype(int).tolist()
    for i, cid in enumerate(ids):
        id_to_emb[int(cid)] = em[i]
print("Mapped embeddings:", len(id_to_emb))

# build text->id map
df = pd.read_csv(CLEANED, usecols=["Complaint ID","Consumer complaint narrative"])
df["Consumer complaint narrative"] = df["Consumer complaint narrative"].astype(str).str.strip()
text_to_id = dict(zip(df["Consumer complaint narrative"].tolist(), df["Complaint ID"].astype(int).tolist()))

# get ids for splits
train_ids = [text_to_id.get(t.strip()) for t in X_train]
test_ids  = [text_to_id.get(t.strip()) for t in X_test]

# filter ids that have embeddings
train_ids_filtered = [cid for cid in train_ids if cid is not None and cid in id_to_emb]
test_ids_filtered  = [cid for cid in test_ids  if cid is not None and cid in id_to_emb]

# save filtered id lists and counts
Path("step7_results").mkdir(parents=True, exist_ok=True)
import json
json.dump({"train_ids_filtered": train_ids_filtered, "test_ids_filtered": test_ids_filtered}, open("step7_results/filtered_ids.json","w"))
print("Filtered counts — train:", len(train_ids_filtered), "test:", len(test_ids_filtered))


Mapped embeddings: 26861
Filtered counts — train: 20932 test: 5232


**Load emb arrays (or rebuild if needed) and assemble y_filtered arrays matching emb order**

In [ ]:
import numpy as np, joblib, json
from pathlib import Path

# load embeddings saved earlier
emb_train = np.load("step7_results/emb_train.npy")
emb_test  = np.load("step7_results/emb_test.npy")
print("emb_train shape:", emb_train.shape, "emb_test shape:", emb_test.shape)

# load filtered ids and label encoder/splits to build y arrays aligned to emb arrays
ids_data = json.load(open("step7_results/filtered_ids.json"))
train_ids_filtered = ids_data["train_ids_filtered"]
test_ids_filtered  = ids_data["test_ids_filtered"]

# load cleaned df to map complaint id -> label
df = pd.read_csv("data/cleaned/cfpb_cleaned_v1.csv", usecols=["Complaint ID","Product"])
df_map = df.set_index("Complaint ID")["Product"].astype(str).to_dict()

# build y arrays in same order as train_ids_filtered/test_ids_filtered
y_train_filtered = np.array([df_map[int(cid)] for cid in train_ids_filtered])
y_test_filtered  = np.array([df_map[int(cid)] for cid in test_ids_filtered])

print("y_train_filtered len:", len(y_train_filtered), "y_test_filtered len:", len(y_test_filtered))
# sanity: lengths should match emb arrays
assert len(y_train_filtered) == emb_train.shape[0], "Mismatch train sizes"
assert len(y_test_filtered)  == emb_test.shape[0],  "Mismatch test sizes"


emb_train shape: (20932, 384) emb_test shape: (5232, 384)
y_train_filtered len: 20932 y_test_filtered len: 5232


**Train emb‑LogReg, evaluate, save artifacts, and copy to Drive**

In [ ]:
import joblib, json, time, shutil
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

OUT = Path("step8_results"); OUT.mkdir(exist_ok=True)
DRIVE_BASE = Path("/content/drive/MyDrive/capstone_artifacts")
DRIVE_RES = DRIVE_BASE/"results"; DRIVE_RES.mkdir(parents=True, exist_ok=True)

# fit
clf = LogisticRegression(max_iter=1000, class_weight="balanced", n_jobs=-1, random_state=42)
t0 = time.time()
clf.fit(emb_train, y_train_filtered)
train_time = time.time() - t0

# predict
y_pred = clf.predict(emb_test)
acc = accuracy_score(y_test_filtered, y_pred)
wf1 = f1_score(y_test_filtered, y_pred, average="weighted")
mf1 = f1_score(y_test_filtered, y_pred, average="macro")
report = classification_report(y_test_filtered, y_pred, zero_division=0)
cm = confusion_matrix(y_test_filtered, y_pred)

# save local
joblib.dump(clf, OUT/"emb_logreg_model.joblib")
open(OUT/"emb_logreg_report.txt","w").write(report)
json.dump({"accuracy":acc,"weighted_f1":wf1,"macro_f1":mf1,"train_time_s":train_time}, open(OUT/"emb_logreg_metrics.json","w"))
np.save(OUT/"emb_logreg_confusion.npy", cm)

# copy to Drive
shutil.copy2(OUT/"emb_logreg_model.joblib", DRIVE_RES/"emb_logreg_model.joblib")
shutil.copy2(OUT/"emb_logreg_report.txt", DRIVE_RES/"emb_logreg_report.txt")
shutil.copy2(OUT/"emb_logreg_metrics.json", DRIVE_RES/"emb_logreg_metrics.json")
shutil.copy2(OUT/"emb_logreg_confusion.npy", DRIVE_RES/"emb_logreg_confusion.npy")

print("Emb-LogReg saved. acc:",acc,"wf1:",wf1,"mf1:",mf1,"train_s:",train_time)


Emb-LogReg saved. acc: 0.588302752293578 wf1: 0.6176380665480808 mf1: 0.4089576067471034 train_s: 10.74069881439209


**Produce models_summary.csv**

In [ ]:
# Produce models_summary.csv from saved metrics
import json, glob, pandas as pd
files = glob.glob("step8_results/*_metrics.json") + glob.glob("step7_results/*_metrics.json") + glob.glob("step7_results/*.json")
rows=[]
for f in files:
    try:
        d=json.load(open(f))
        name = f.split("/")[-1].replace("_metrics.json","").replace(".json","")
        rows.append({
            "model": name,
            "accuracy": d.get("accuracy"),
            "weighted_f1": d.get("weighted_f1"),
            "macro_f1": d.get("macro_f1"),
            "train_time_s": d.get("train_time_s") or d.get("train_time") or d.get("train_time_s")
        })
    except:
        continue
df = pd.DataFrame(rows).drop_duplicates(subset=["model"]).sort_values("weighted_f1", ascending=False)
df.to_csv("step8_results/models_summary.csv", index=False)
print("Saved step8_results/models_summary.csv")
print(df.head(10))


Saved step8_results/models_summary.csv
          model  accuracy  weighted_f1  macro_f1  train_time_s
0    emb_logreg  0.588303     0.617638  0.408958     10.740699
1  filtered_ids       NaN          NaN       NaN           NaN


**Copy final artifacts to Drive and compute MD5 for key files**

In [ ]:
# Copy artifacts to Drive and record MD5s
from pathlib import Path
import shutil, hashlib, json
DR = Path("/content/drive/MyDrive/capstone_artifacts")
DR.mkdir(parents=True, exist_ok=True)

# Ensure target folders
(DR/"artifacts").mkdir(parents=True, exist_ok=True)
(DR/"artifacts"/"embeddings").mkdir(parents=True, exist_ok=True)
(DR/"models").mkdir(parents=True, exist_ok=True)
(DR/"results").mkdir(parents=True, exist_ok=True)
(DR/"logs").mkdir(parents=True, exist_ok=True)

# copy folders/files
def copy_if_exists(src, dst):
    src=Path(src)
    if src.exists():
        if src.is_dir():
            shutil.copytree(src, dst, dirs_exist_ok=True)
        else:
            shutil.copy2(src, dst)
        print("Copied:", src, "->", dst)
    else:
        print("Missing, skipped:", src)

copy_if_exists("artifacts/embeddings", DR/"artifacts"/"embeddings")
copy_if_exists("artifacts/faiss", DR/"artifacts"/"faiss")
copy_if_exists("models", DR/"models")
copy_if_exists("step8_results", DR/"results"/"step8_results")
copy_if_exists("step7_results", DR/"results"/"step7_results")

# compute md5 helper
def md5(p):
    h=hashlib.md5()
    with open(p,"rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

meta = {}
key_files = [
    DR/"data"/"cleaned"/"cfpb_cleaned_step5.csv",
    DR/"artifacts"/"faiss"/"cfpb_ivf.index",
    DR/"results"/"step8_results"/"models_summary.csv",
    DR/"artifacts"/"splits"/"split_and_le_step7.joblib"
]
for p in key_files:
    if p.exists():
        meta[str(p)] = md5(str(p))
    else:
        meta[str(p)] = None

# save meta
Path("step8_results").mkdir(parents=True, exist_ok=True)
json.dump(meta, open("step8_results/artifact_md5s.json","w"), indent=2)
shutil.copy2("step8_results/artifact_md5s.json", DR/"logs"/"artifact_md5s_step8.json")
print("Copied artifacts and saved MD5s to Drive logs.")


Copied: artifacts/embeddings -> /content/drive/MyDrive/capstone_artifacts/artifacts/embeddings
Copied: artifacts/faiss -> /content/drive/MyDrive/capstone_artifacts/artifacts/faiss
Copied: models -> /content/drive/MyDrive/capstone_artifacts/models
Copied: step8_results -> /content/drive/MyDrive/capstone_artifacts/results/step8_results
Copied: step7_results -> /content/drive/MyDrive/capstone_artifacts/results/step7_results
Copied artifacts and saved MD5s to Drive logs.


**Create README_step8.txt and short submission summary**

In [ ]:
# Create README and one-page summary
from pathlib import Path
import shutil, json
DR = Path("/content/drive/MyDrive/capstone_artifacts")

readme = f"""
Step 8 — Scaled Prototype (Capstone)
-----------------------------------
Artifacts (Drive):
- Cleaned data: {DR}/data/cleaned/cfpb_cleaned_step5.csv
- Embeddings: {DR}/artifacts/embeddings/
- FAISS index: {DR}/artifacts/faiss/
- Models: {DR}/models/
- Results: {DR}/results/step8_results/
- Logs/metadata: {DR}/logs/

Primary notes:
- Primary metric: weighted F1.
- Best immediate baseline: TF-IDF + LogisticRegression (see models_summary.csv).
- Embeddings generated with all-MiniLM-L6-v2 and FAISS IVF index built for retrieval.
- Experiment metadata and checksums stored in logs/experiment_meta_step8.json and artifact_md5s_step8.json.

Reproduce (high level):
1) Mount Drive in Colab.
2) Run notebooks/08_scale_prototype.ipynb cells in order.
3) Artifacts are persisted to the Drive paths above.

"""
Path("step8_results").mkdir(parents=True, exist_ok=True)
open("step8_results/README_step8.txt","w").write(readme)
shutil.copy2("step8_results/README_step8.txt", DR/"logs"/"README_step8.txt")
print("README_step8.txt created and copied to Drive logs.")


README_step8.txt created and copied to Drive logs.


Summary — Findings and Business Explanation

Project goal

Build a compliance‑aware customer‑intelligence pipeline that classifies CFPB consumer complaint narratives, supports semantic retrieval, and generates evidence‑grounded insights at scale.
Key technical findings

Data: A cleaned canonical dataset (cfpb_cleaned_step5.csv) of 26,861 complaint narratives was used; splits and provenance (MD5, seed) were saved for reproducibility.
Baseline performance: TF‑IDF + Logistic Regression (class_weight="balanced") produced the best immediate result (accuracy ≈ 0.71, weighted F1 ≈ 0.71). This model is fast, compact, and interpretable.
Embedding experiments: Sentence‑Transformer (all‑MiniLM‑L6‑v2) embeddings + Logistic Regression yielded weighted F1 ≈ 0.62; embeddings + LightGBM improved to ≈ 0.68. Embeddings capture semantics but require tuning/ensembling to match the TF‑IDF baseline on this task.
Retrieval: Batched embeddings were generated and an on‑disk FAISS IVF index was built. Retrieval results for sample queries returned relevant complaint records, enabling evidence‑grounded outputs.
Scalability: Implemented chunked ETL with PII redaction, out‑of‑core training (HashingVectorizer + SGD), batched embedding generation with checkpointing, incremental FAISS index building, and artifact/version logging.
Business interpretation

Immediate value: Deploy TF‑IDF + Logistic Regression as an automated triage model for high‑volume, common complaint categories. It reduces manual reviews and provides explainable predictions useful to compliance and operations teams.
Explainability & auditability: The TF‑IDF model and the FAISS retrieval index enable traceability — each automated decision can be linked to redacted complaint IDs and source text for audits.
Risk & limitations: Macro F1 is much lower than weighted F1, indicating poor performance on minority/rare categories. Rare but high‑impact complaints (e.g., fraud, regulatory violations) risk being misclassified by automated workflows.
Mitigation: Flag low‑confidence predictions and rare‑class outputs for human review; retain full audit logs (retrieved IDs, prompts, model outputs) for compliance review.
Recommended next steps

Productionize the TF‑IDF triage model with human‑in‑the‑loop review for low‑confidence or rare categories.
Improve minority‑class performance via targeted strategies: hyperparameter tuning, ensembling TF‑IDF and embedding models, class‑aware sampling, or focal loss.
Harden RAG pipeline: index only redacted text, require explicit citation of complaint IDs in generated summaries, and persist retrieval/generation logs for audits.
For larger scale or higher reliability, move artifacts to cloud storage (S3/GCS) and use a managed vector DB (Milvus/Pinecone) and distributed training (SparkML or TF/PyTorch on GPU) where needed.
Conclusion

The prototype yields an immediately useful, interpretable baseline (TF‑IDF + LogReg) that can be deployed for operational triage. To safely scale for regulatory and high‑impact use cases, prioritize auditability, human oversight for uncertain cases, and targeted improvements for rare but important complaint categories.

# **README**

Invalid Notebook
There was an error rendering your Notebook: the 'state' key is missing from 'metadata.widgets'. Add 'state' to each, or remove 'metadata.widgets'.
Using nbformat v5.10.4 and nbconvert v7.16.6